# v2 研究：滚动季度验证与文本特征比较

把本文件放入 `~/explaining-markets-research/notebooks/`，选择研究项目的 `.venv`，从上到下运行。
复用已有 `data/archive/`；不下载数据，不需要 OpenAI key，不提交预测、不部署或覆盖 v1 模型。
若缺少依赖，在研究项目终端运行 `python -m pip install scikit-learn pandas numpy scipy joblib`。

实验预先固定为 **2 种特征 × 6 个 alpha × 3 个验证季度**：
- word：与 v1 相同的 unigram + bigram TF-IDF。
- word_char：在 word 上增加单词边界内 3–5 字符片段，尝试共享 growth/growing 等词形的信息；是否有用以验证结果为准。
- alpha：1、10、100、300、1000、3000。word / 100 是 v1 配置对照，**每折重新训练**，绝不加载已训练过验证季度的线上模型。
- 2025Q4 → 2026Q1；2025Q4–2026Q1 → 2026Q2；2025Q4–2026Q2 → 2026Q3。

**这三个季度均作为开发验证集**。Q3 已经反复看过，不能再称为独立测试集。
不使用未结束的 2026Q4；未来独立评估应从新策略冻结后尚未用于调参的事件开始。
事后档案可能包含后补/修订信息，因此这是历史研究，不是已证明的 point-in-time 回测。

主要指标是官方 `delta_r_squared_imputed`：在财报 surprise 的解释力之上，预测增加多少解释力。
MSE / Spearman 仅作诊断；季度平均 ΔR² 是选模型用的研究摘要，不是跨季度官方比赛分数。
官方实现参考：https://github.com/explaining-markets/examples/blob/501bd31/src/examples/scoring.py


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import hashlib, inspect, json, math, re, sys
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.pipeline import FeatureUnion
from sklearn.metrics import mean_squared_error
from examples.archive import load_archive
from examples.summary import facts_from_disclosure
from examples import scoring
from examples.scoring import percentile_ranks, add_percentiles, score_submission

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'data'/'archive').is_dir()), None)
if REPO is None:
    raise RuntimeError('请放在研究项目 notebooks 文件夹，并先下载历史档案。')
ARCHIVE_DIR = REPO/'data'/'archive'
archive_files = sorted(ARCHIVE_DIR.rglob('*.jsonl.gz'))
if not archive_files:
    raise RuntimeError('缺少 *.jsonl.gz 历史档案。')
df = load_archive(ARCHIVE_DIR)
QUARTERS = ['2025Q4', '2026Q1', '2026Q2', '2026Q3']
ALPHAS = [1.0, 10.0, 100.0, 300.0, 1000.0, 3000.0]
FAMILIES = ['word', 'word_char']
BASELINE = 'word_a100'
print('档案行数：', len(df), ' sklearn：', sklearn.__version__)


## 1. 清洗、标签与共同评分样本

只用财报事实作为输入；收益、surprise、官方模型预测都不会进入特征。
每季度先对完整有效收益横截面算 y，再对有 surprise 的记录算 surprise_pct。
文本缺失仍保留，预测 0.5；缺少 surprise 的行可以参与训练，但不进入官方条件回归评分。
同一事件/股票的冲突重复记录会停止运行。


In [ ]:
def facts_text(record):
    disclosure = record.get('disclosure')
    if not isinstance(disclosure, dict):
        return ''
    facts = facts_from_disclosure(record)
    return '\n'.join(f.strip() for f in facts if isinstance(f, str) and f.strip())

def make_outcomes(frame):
    rows, skipped = [], Counter()
    for rec in frame.to_dict('records'):
        if rec.get('event_type') != 'EARNINGS_RELEASE':
            skipped['not_earnings'] += 1
            continue
        if not isinstance(rec.get('event_id'), str):
            skipped['missing_event_id'] += 1
            continue
        quarter = rec.get('quarter')
        if not isinstance(quarter, str) or not re.fullmatch(r'\d{4}Q[1-4]', quarter):
            skipped['invalid_quarter'] += 1
            continue
        assets, returns = rec.get('focal_assets'), rec.get('event_returns')
        if not isinstance(assets, list) or not isinstance(returns, dict):
            skipped['missing_assets_or_returns'] += 1
            continue
        text = facts_text(rec)
        metrics = rec.get('metrics')
        es = metrics.get('earnings_surprise') if isinstance(metrics, dict) else None
        sv = es.get('surprise') if isinstance(es, dict) and es.get('surprise_status') == 'ok' else None
        surprise = float(sv) if isinstance(sv, (int, float)) and not isinstance(sv, bool) and math.isfinite(sv) else np.nan
        for asset in assets:
            if not isinstance(asset, dict):
                continue
            ticker = asset.get('identifier_value')
            if not isinstance(ticker, str) or not ticker:
                continue
            leg = returns.get(ticker)
            if not isinstance(leg, dict) or leg.get('return_status') != 'ok':
                skipped['invalid_return_status'] += 1
                continue
            value = leg.get('car1')
            if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
                skipped['invalid_car1'] += 1
                continue
            rows.append(dict(event_id=rec['event_id'], ticker=ticker, quarter=quarter,
                             event_datetime=rec.get('event_datetime'), text=text, car1=float(value), surprise=surprise))
    if not rows:
        raise RuntimeError('没有找到有效财报收益，请检查是否使用了完整档案。')
    out = pd.DataFrame(rows)
    keys = ['event_id', 'ticker']
    duplicates = out[out.duplicated(keys, keep=False)]
    if not duplicates.empty:
        conflict = duplicates.groupby(keys)[['quarter', 'car1', 'text', 'event_datetime', 'surprise']].nunique(dropna=False)
        if (conflict > 1).any().any():
            raise RuntimeError('重复事件记录存在冲突，请检查档案版本。')
        out = out.drop_duplicates(keys).copy()
    out['event_datetime'] = pd.to_datetime(out['event_datetime'], utc=True, errors='coerce')
    if out['event_datetime'].isna().any():
        raise RuntimeError('存在无效事件时间，请先检查。')
    # 官方每季度百分位变换；不是 pandas rank(pct=True)。
    out['y'] = out.groupby('quarter')['car1'].transform(lambda s: percentile_ranks(s.tolist()))
    out['has_text'] = out['text'].str.contains(r'[A-Za-z]{2}', regex=True)
    return out.sort_values(['event_datetime', 'event_id', 'ticker']).reset_index(drop=True), skipped


In [ ]:
data, skipped = make_outcomes(df)
data = pd.concat([add_percentiles(g.drop(columns='y'))
                  for _, g in data.groupby('quarter')], ignore_index=True)
now = datetime.now(timezone.utc)
current_quarter = f'{now.year}Q{(now.month-1)//3+1}'
if any(q >= current_quarter for q in QUARTERS):
    raise RuntimeError('预定实验包含未结束季度，请检查日期。')
missing = set(QUARTERS) - set(data.quarter)
if missing:
    raise RuntimeError(f'缺少实验季度：{sorted(missing)}')
print('排除实验范围外季度：', sorted(set(data.quarter)-set(QUARTERS)))
data = data[data.quarter.isin(QUARTERS)].copy()
print('清洗跳过：', dict(skipped))
display(data.groupby('quarter').agg(rows=('y','size'), with_text=('has_text','sum'),
                                   with_surprise=('surprise_pct','count')))


## 2. 特征与评分函数

每折 TF-IDF 只看该折训练文本；同一折的词表复用于不同 alpha，减少重复计算。
word_char 固定拼接权重 word=1、char=0.5，不在本轮额外搜索权重。
预测裁剪到 [0,1]，与部署要求一致。不会为了让数字“更有信心”而事后放大预测。


In [ ]:
def make_features(family):
    word = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=30000,
                           sublinear_tf=True, dtype=np.float64)
    if family == 'word':
        return word
    char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), min_df=3,
                           max_features=30000, sublinear_tf=True, dtype=np.float64)
    return FeatureUnion([('word', word), ('char', char)],
                         transformer_weights={'word':1.0, 'char':0.5}, n_jobs=1)

def evaluate(part, pred):
    pred = np.asarray(pred, dtype=float)
    if len(pred) != len(part) or not np.isfinite(pred).all():
        raise ValueError('预测长度不一致或存在非有限数。')
    frame = part[['y','surprise_pct']].copy()
    frame['prediction'] = pred
    score = score_submission(frame, 'prediction')
    delta = score.get('delta_r_squared_imputed')
    if delta is None or not np.isfinite(delta):
        raise RuntimeError('官方评分无定义，请检查 surprise 分布和有效样本数。')
    return dict(delta_R2=float(delta), n_obs=score['n_obs'],
                MSE=float(mean_squared_error(part.y, pred)),
                Spearman=pd.Series(part.y.to_numpy()).corr(pd.Series(pred), method='spearman')
                    if np.std(pred)>1e-12 else np.nan,
                prediction_std=float(np.std(pred)),
                missing_text_count=int((~part.has_text).sum()))


## 3. 运行 36 次 Ridge 拟合

耗时取决于机器和文本长度。每完成一个季度/特征组合会打印进度。
不要把已看见的结果当作新的独立测试，或反复扩展搜索直到某一季度特别漂亮。


In [ ]:
fold_rows, prediction_rows = [], []
for i, valid_q in enumerate(QUARTERS[1:], start=1):
    train_qs = QUARTERS[:i]
    train = data[data.quarter.isin(train_qs) & data.has_text].copy()
    valid = data[data.quarter.eq(valid_q)].copy()
    assert set(train.event_id).isdisjoint(valid.event_id)
    assert train.event_datetime.max() < valid.event_datetime.min()
    if len(train)<30 or valid.surprise_pct.notna().sum()<30:
        raise RuntimeError('训练或有效评分样本过少。')
    valid_mask = valid.has_text.to_numpy()
    constant_score = evaluate(valid, np.full(len(valid), 0.5))
    fold_rows.append(dict(candidate='constant', family='constant', alpha=np.nan,
                          quarter=valid_q, train_quarters=','.join(train_qs), **constant_score))
    for family in FAMILIES:
        features = make_features(family)
        X_train = features.fit_transform(train.text)
        X_valid = features.transform(valid.text)  # transform only, never fit
        for alpha in ALPHAS:
            ridge = Ridge(alpha=alpha, solver='lsqr')
            ridge.fit(X_train, train.y)
            pred = np.clip(ridge.predict(X_valid), 0.0, 1.0)
            pred[~valid_mask] = 0.5
            candidate = f'{family}_a{alpha:g}'
            fold_rows.append(dict(candidate=candidate, family=family, alpha=alpha,
                                  quarter=valid_q, train_quarters=','.join(train_qs),
                                  **evaluate(valid, pred)))
            rows = valid[['event_id','ticker','quarter','y','surprise_pct','has_text']].copy()
            rows['candidate'], rows['prediction'] = candidate, pred
            prediction_rows.append(rows)
        print('完成', ','.join(train_qs), '→', valid_q, family, '特征数', X_train.shape[1], flush=True)
        del features, X_train, X_valid, ridge
fold_results = pd.DataFrame(fold_rows)
oof_predictions = pd.concat(prediction_rows, ignore_index=True)
print('完成。以下均为历史开发验证结果。')


## 4. 比较平均表现和跨季度稳定性

`mean_gain_vs_v1`：每季度 ΔR² 相对重新训练的 v1 配置的差值，再等权平均。
`worst_gain_vs_v1`：最差季度差值；`quarters_beating_v1`：胜过对照的季度数。
先按平均 ΔR² 排序，再看逐季度表现。候选微小领先或只靠一个季度，应继续收集证据。
只有三折，不能据此声称“统计显著提升”。本 notebook 不自动导出/部署胜出模型。


In [ ]:
quarter_scores = fold_results.pivot(index='candidate', columns='quarter', values='delta_R2')
gains = quarter_scores.sub(quarter_scores.loc[BASELINE], axis=1)
summary = pd.DataFrame({
    'mean_delta_R2':quarter_scores.mean(axis=1),
    'worst_delta_R2':quarter_scores.min(axis=1),
    'mean_gain_vs_v1':gains.mean(axis=1),
    'worst_gain_vs_v1':gains.min(axis=1),
    'quarters_beating_v1':(gains>1e-10).sum(axis=1),
}).join(quarter_scores).sort_values('mean_delta_R2', ascending=False)
display(summary)
display(fold_results[['candidate','quarter','n_obs','delta_R2','MSE','Spearman','prediction_std']])
best_candidate = summary.drop(index='constant').index[0]
print('待进一步验证的候选：', best_candidate)
print('此排名不等于独立测试结果；线上 v1 保持当前版本。')


## 5. 保存研究结果

仅写入研究项目新建的 `artifacts/strategy_v2_research/<时间戳>/`。
不保存原始财报文本、签名 URL 或密钥。记录源档案/评分代码哈希以便复现。
比较 summary 和逐季度结果，再预先固定下一轮实验。
字符特征候选若胜出，还需适配当前部署的模型结构校验；不要直接替换线上 joblib。


In [ ]:
SAVE_RESULTS = True
if SAVE_RESULTS:
    run_dir = REPO/'artifacts'/'strategy_v2_research'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    run_dir.mkdir(parents=True, exist_ok=False)
    summary.to_csv(run_dir/'summary.csv')
    fold_results.to_csv(run_dir/'fold_results.csv', index=False)
    oof_predictions.to_csv(run_dir/'oof_predictions.csv', index=False)
    def sha256(path):
        digest = hashlib.sha256()
        with path.open('rb') as stream:
            for chunk in iter(lambda:stream.read(1024*1024), b''):
                digest.update(chunk)
        return digest.hexdigest()
    scoring_path = Path(inspect.getfile(scoring))
    manifest = dict(quarters=QUARTERS, alphas=ALPHAS, families=FAMILIES,
                    baseline=BASELINE, selected_for_further_validation=best_candidate,
                    sklearn_version=sklearn.__version__, python=sys.version,
                    scoring_sha256=sha256(scoring_path),
                    archives={str(p.relative_to(REPO)):sha256(p) for p in archive_files},
                    evaluation='retrospective development validation; not independent holdout',
                    deployed=False)
    (run_dir/'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print('研究结果保存到：', run_dir)
